# Using state to persist data

To store data in a session state, you need to do three things:

1.  Pass in an _extra parameter_ into your function, which represents the state of the interface.
2.  At the end of the function, return the updated value of the state as an _extra return value_.
3.  Add the ‘state’ input and ‘state’ output components when creating your `Interface`.

In [ ]:
import random
import gradio as gr

def chat(message, history):
    history = history or []
    if message.startswith("How many"):
        response = random.randint(1, 10)
    elif message.startswith("How"):
        response = random.choice(["Great", "Good", "Okay", "Bad"])
    elif message.startswith("Where"):
        response = random.choice(["Here", "There", "Somewhere"])
    else:
        response = "I don't know"
    history.append((message, response))
    return response + " | ".join(map(lambda x: x[0], history)), history


iface = gr.Interface(
    chat,
    ["text", "state"],
    ["text", "state"],
    
)
iface.launch()

# Using interpretation to understand predictions

- This allows your users to understand what parts of the input are responsible for the output.
- Take a look at the simple interface below which shows an image classifier that also includes interpretation.

In [ ]:
import requests
import torchvision.models as models
import gradio as gr

# load the model
inception_net = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Download human-readable labels for ImageNet.
response = requests.get("https://git.io/JJkYN")
labels = response.text.split("\n")[:-1]

In [ ]:
import torch
test_output = inception_net(torch.zeros(1,3,224,224))
assert len(labels) == test_output.shape[-1]

In [ ]:
def classify_image(inp):
    tensor_inp = torch.tensor(inp, dtype=torch.float32)\
        .permute(2,0,1).unsqueeze(0)
    prediction = inception_net(tensor_inp).flatten()
    return {labels[i]: float(prediction[i]) for i in range(1000)}


image = gr.Image(height=224, width=224)
label = gr.Label(num_top_classes=3)

title = "Gradio Image Classifiction + Interpretation Example"
gr.Interface(
    fn=classify_image, inputs=image, outputs=label,
    title=title,
    # interpretation="default", --> this is deprecated
).launch()